# TSLA Momentum Model — Notebook 3: Performance Analysis & Validation

Contents:

1. Full performance summary (portfolio-level and trade-level metrics) over the entire backtest.
2. Supporting charts: equity curve, drawdown, and entries/exits against price.
3. An in-sample / out-of-sample split. Parameters were fixed *a priori* in Notebook 1, but reporting performance separately on a held-out period tests whether the full-period result is reproducible.

In [ ]:
import sys
import os

# Resolve the src/ folder regardless of whether the working directory is
# the notebook's own folder (Jupyter/Colab default) or the project root
# (which some IDEs, including PyCharm's Jupyter integration, may use instead).
for _candidate in ("../src", "src"):
    if os.path.isdir(_candidate):
        sys.path.insert(0, _candidate)
        break
else:
    raise RuntimeError(
        "Could not locate the src/ folder from the current working directory: "
        + os.getcwd()
    )

import pandas as pd

from backtest import run_backtest, trades_to_frame
import metrics as m
from plotting import plot_equity_curves, plot_drawdown, plot_trades_on_price

df = pd.read_pickle("tsla_signals.pkl")
equity_curve = pd.read_pickle("equity_curve.pkl")
trade_log = pd.read_pickle("trade_log.pkl")

## 1. Full-period performance summary

In [ ]:
summary, trade_summary = m.summarize(equity_curve, trade_log)
print("Portfolio-level metrics:")
display(summary)
print()
print("Trade-level metrics:")
display(trade_summary)

## 2. Charts

In [ ]:
fig1 = plot_equity_curves(equity_curve)

In [ ]:
fig2 = plot_drawdown(equity_curve)

In [ ]:
fig3 = plot_trades_on_price(df, trade_log)

## 3. In-sample / out-of-sample split

The strategy's parameters (EMA lengths, RSI band, trend window, ATR stop multiplier) were fixed in Notebook 1 using standard conventions, before any backtest result was available. The data is split at a fixed date and the same parameters and the same backtest engine are rerun independently on each half. No re-fitting takes place on the out-of-sample period.

In [ ]:
SPLIT_DATE = "2022-01-01"

df_in_sample = df[df.index < SPLIT_DATE].copy()
df_out_of_sample = df[df.index >= SPLIT_DATE].copy()

print(f"In-sample:     {df_in_sample.index[0].date()} to {df_in_sample.index[-1].date()}  ({len(df_in_sample)} days)")
print(f"Out-of-sample: {df_out_of_sample.index[0].date()} to {df_out_of_sample.index[-1].date()}  ({len(df_out_of_sample)} days)")

In [ ]:
results = {}
for label, segment in [("In-Sample", df_in_sample), ("Out-of-Sample", df_out_of_sample)]:
    eq, tr = run_backtest(segment)
    tr_df = trades_to_frame(tr)
    port = m.portfolio_metrics(eq, "equity")
    bh = m.portfolio_metrics(eq, "buy_hold_equity")
    trades_m = m.trade_metrics(tr_df)
    results[label] = {**{f"strategy_{k}": v for k, v in port.items()},
                       **{f"buy_hold_{k}": v for k, v in bh.items()},
                       **trades_m}

comparison = pd.DataFrame(results)
comparison

## 4. Reading the comparison table

The question is whether the out-of-sample Sharpe, Sortino, and drawdown profile is consistent with the in-sample period, or whether performance deteriorates outside the window the design was based on. Both outcomes are informative; the split exists to make the distinction visible rather than reporting only the full-period figure.